In [2]:
import sys, json
from pathlib import Path
import numpy as np
import torch
import matplotlib.pyplot as plt
import matplotlib
from PIL import Image
from torchvision import transforms
from collections import Counter
from matplotlib.lines import Line2D
import matplotlib.patches as mpatches

matplotlib.rcParams.update({'font.size': 11, 'figure.figsize': (16, 8)})

repo_root = Path.cwd()
if not (repo_root / "src").exists():
    repo_root = repo_root.parent
sys.path.insert(0, str(repo_root))

from src.indexing.base import query_index, neighbor_vectors, load_index, build_index
from src.indexing.image_index import load_image_index_artifacts
from src.smoothing.manifold import ManifoldSmoother
from src.smoothing.pca import fit_local_pca, LocalPCA
from src.models.VAE import ConvVAE, load_checkpoint as load_vae_checkpoint
from src.models.resnet import build_resnet_classifier

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Imports OK — device: {device}")

Imports OK — device: cpu


## 1. Configure Paths

**Set these to your server paths.** Everything else derives from here.

In [3]:
# ============================================================
# >>> SET THESE TO YOUR SERVER PATHS <<<
# ============================================================
OUTPUT_ROOT = repo_root / "output"  # or Path("/your/server/output")

DATASET_NAME = "celeba"  # "celeba" or "celebahq"
IMAGE_SIZE   = 128       # must match your trained models (64 or 128)
LATENT_DIM   = 128       # must match your VAE

# Derived paths
BASE_DIR = OUTPUT_ROOT / "smile_classification" / DATASET_NAME

# Indexes (built from train split)
PIXEL_INDEX_DIR  = BASE_DIR / "index" / "pixel"  / "annoy" / "euclidean"
LATENT_INDEX_DIR = BASE_DIR / "index" / "latent" / "annoy" / "euclidean"

# Model checkpoints
CLASSIFIER_CKPT = OUTPUT_ROOT / "pretrained_model" / f"smile_resnet_{DATASET_NAME}" / "best.pt"
VAE_CKPT        = OUTPUT_ROOT / "pretrained_model" / f"vae_{DATASET_NAME}_{IMAGE_SIZE}" / "best.pt"

# CelebA images — needed to load test samples for visualization
# Set this to where your CelebA images actually live
CELEBA_IMAGE_DIR = repo_root / "input" / "datasets" / "celebAHQ"  # adjust as needed

print("Paths configured:")
for name, p in [("BASE_DIR", BASE_DIR), ("PIXEL_INDEX", PIXEL_INDEX_DIR),
                 ("LATENT_INDEX", LATENT_INDEX_DIR), ("CLASSIFIER", CLASSIFIER_CKPT),
                 ("VAE", VAE_CKPT)]:
    exists = "✓" if Path(p).exists() else "✗ NOT FOUND"
    print(f"  {name:15s}: {p}  [{exists}]")

Paths configured:
  BASE_DIR       : d:\THESIS\RandomizedSmothingmanifold\output\smile_classification\celeba  [✓]
  PIXEL_INDEX    : d:\THESIS\RandomizedSmothingmanifold\output\smile_classification\celeba\index\pixel\annoy\euclidean  [✓]
  LATENT_INDEX   : d:\THESIS\RandomizedSmothingmanifold\output\smile_classification\celeba\index\latent\annoy\euclidean  [✓]
  CLASSIFIER     : d:\THESIS\RandomizedSmothingmanifold\output\pretrained_model\smile_resnet_celeba\best.pt  [✓]
  VAE            : d:\THESIS\RandomizedSmothingmanifold\output\pretrained_model\vae_celeba_128\best.pt  [✓]


## 2. Load Models & Indexes

In [4]:
# ── Smile Classifier (ResNet, binary: sigmoid output) ──
classifier = build_resnet_classifier(name="resnet18", pretrained=False, num_classes=1)
ckpt = torch.load(CLASSIFIER_CKPT, map_location=device)
state = ckpt.get("model_state_dict", ckpt.get("state_dict", ckpt))
classifier.load_state_dict(state)
classifier.to(device).eval()
print(f"Classifier loaded: {CLASSIFIER_CKPT.name}")

# ── VAE ──
vae = ConvVAE(in_channels=3, image_size=IMAGE_SIZE, latent_dim=LATENT_DIM)
load_vae_checkpoint(vae, VAE_CKPT, device=device)
vae.to(device).eval()
print(f"VAE loaded: {VAE_CKPT.name}  (latent_dim={LATENT_DIM}, image_size={IMAGE_SIZE})")

# ── Image transform (must match training) ──
CELEBA_MEAN = [0.5, 0.5, 0.5]
CELEBA_STD  = [0.5, 0.5, 0.5]

img_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(CELEBA_MEAN, CELEBA_STD),
])

RuntimeError: Error(s) in loading state_dict for ResNet:
	Missing key(s) in state_dict: "conv1.weight", "bn1.weight", "bn1.bias", "bn1.running_mean", "bn1.running_var", "layer1.0.conv1.weight", "layer1.0.bn1.weight", "layer1.0.bn1.bias", "layer1.0.bn1.running_mean", "layer1.0.bn1.running_var", "layer1.0.conv2.weight", "layer1.0.bn2.weight", "layer1.0.bn2.bias", "layer1.0.bn2.running_mean", "layer1.0.bn2.running_var", "layer1.1.conv1.weight", "layer1.1.bn1.weight", "layer1.1.bn1.bias", "layer1.1.bn1.running_mean", "layer1.1.bn1.running_var", "layer1.1.conv2.weight", "layer1.1.bn2.weight", "layer1.1.bn2.bias", "layer1.1.bn2.running_mean", "layer1.1.bn2.running_var", "layer2.0.conv1.weight", "layer2.0.bn1.weight", "layer2.0.bn1.bias", "layer2.0.bn1.running_mean", "layer2.0.bn1.running_var", "layer2.0.conv2.weight", "layer2.0.bn2.weight", "layer2.0.bn2.bias", "layer2.0.bn2.running_mean", "layer2.0.bn2.running_var", "layer2.0.downsample.0.weight", "layer2.0.downsample.1.weight", "layer2.0.d

In [5]:
# ── Load pre-built Annoy indexes + vectors ──

# Pixel index
pixel_vectors, pixel_meta = load_image_index_artifacts(PIXEL_INDEX_DIR)
pixel_dim = pixel_vectors.shape[1]
pixel_index = load_index(
    dim=pixel_dim,
    index_path=str(PIXEL_INDEX_DIR / "index.ann"),
    backend="annoy", metric="euclidean",
)
pixel_index.vectors = pixel_vectors
print(f"Pixel index: {pixel_vectors.shape[0]} vectors, dim={pixel_dim}")

# Latent index
latent_vectors, latent_meta = load_image_index_artifacts(LATENT_INDEX_DIR)
latent_dim = latent_vectors.shape[1]
latent_index = load_index(
    dim=latent_dim,
    index_path=str(LATENT_INDEX_DIR / "index.ann"),
    backend="annoy", metric="euclidean",
)
latent_index.vectors = latent_vectors
print(f"Latent index: {latent_vectors.shape[0]} vectors, dim={latent_dim}")

FileNotFoundError: [Errno 2] No such file or directory: 'd:\\THESIS\\RandomizedSmothingmanifold\\output\\smile_classification\\celeba\\index\\pixel\\annoy\\euclidean\\image_vectors.npz'

## 3. Helper Functions

In [ ]:
# ── Tensor ↔ Image conversion ──

def tensor_to_pil(tensor, mean=CELEBA_MEAN, std=CELEBA_STD):
    """Denormalize (C,H,W) tensor → PIL Image."""
    img = tensor.clone().detach().cpu()
    for c in range(3):
        img[c] = img[c] * std[c] + mean[c]
    img = img.clamp(0, 1)
    return Image.fromarray((img.permute(1, 2, 0).numpy() * 255).astype(np.uint8))


def flat_pixel_to_pil(flat_vec, image_size=IMAGE_SIZE):
    """Convert flattened pixel vector (C*H*W,) → PIL via denormalize."""
    t = torch.from_numpy(flat_vec.reshape(3, image_size, image_size)).float()
    return tensor_to_pil(t)


def latent_to_pil(z_vec):
    """Decode latent vector → PIL Image via VAE."""
    with torch.no_grad():
        z_t = torch.from_numpy(z_vec[None, :]).to(device=device, dtype=torch.float32)
        x_hat = vae.decode(z_t).squeeze(0).cpu()
    return tensor_to_pil(x_hat)


# ── Classifier ──

def classify_image_tensor(img_tensor):
    """Classify a single (C,H,W) tensor. Returns ('Smile', prob) or ('No Smile', prob)."""
    with torch.no_grad():
        logit = classifier(img_tensor.unsqueeze(0).to(device)).squeeze()
        prob = torch.sigmoid(logit).item()
    label = "Smile" if prob > 0.5 else "No Smile"
    return label, prob


def classify_pixel_vectors(vecs):
    """Classify flattened pixel vectors (N, C*H*W). Returns list of ('Smile'/'No Smile', prob)."""
    if vecs.ndim == 1:
        vecs = vecs.reshape(1, -1)
    results = []
    for v in vecs:
        t = torch.from_numpy(v.reshape(3, IMAGE_SIZE, IMAGE_SIZE)).float()
        results.append(classify_image_tensor(t))
    return results


def classify_latent_vectors(vecs):
    """Classify latent vectors by decoding through VAE first. Returns list of ('Smile'/'No Smile', prob)."""
    if vecs.ndim == 1:
        vecs = vecs.reshape(1, -1)
    results = []
    with torch.no_grad():
        for v in vecs:
            z_t = torch.from_numpy(v[None, :]).to(device=device, dtype=torch.float32)
            x_hat = vae.decode(z_t).squeeze(0)
            results.append(classify_image_tensor(x_hat.cpu()))
    return results


def project_to_local_pca_2d(points, pca):
    centered = points - pca.mean
    return centered @ pca.evecs[:, :2]


print("Helpers defined.")

## 4. Select Test Samples (Smile + No Smile)

In [ ]:
# We pick a few images from the test set for visualization.
# If you have a results.csv from certification, load from there;
# otherwise, load a few images directly.

# Option A: Load from certification results.csv (has idx, label, pred)
results_csv = BASE_DIR / "certify" / "latent_manifold" / "sigma_0_50" / "results.csv"

if results_csv.exists():
    import pandas as pd
    df = pd.read_csv(results_csv)
    print(f"Loaded {len(df)} certified samples from {results_csv.name}")
    print(df.head())
    # Pick one smile and one no-smile that were correctly classified
    correct = df[df['correct'] == True]
    smile_rows = correct[correct['label'] == 1]
    no_smile_rows = correct[correct['label'] == 0]
    rng = np.random.default_rng(42)
    selected_idxs = []
    if len(smile_rows) > 0:
        selected_idxs.append(smile_rows.iloc[rng.integers(len(smile_rows))]['idx'])
    if len(no_smile_rows) > 0:
        selected_idxs.append(no_smile_rows.iloc[rng.integers(len(no_smile_rows))]['idx'])
    print(f"Selected test indices: {selected_idxs}")
else:
    # Option B: Just pick indices into the index vectors (train set)
    # and treat them as samples. The index was built from train.
    rng = np.random.default_rng(42)
    n_total = pixel_vectors.shape[0]
    selected_idxs = rng.choice(n_total, size=4, replace=False).tolist()
    print(f"No results.csv found. Using random train indices: {selected_idxs}")
    print("(Classify them to determine smile/no-smile)")

## 5. Visualization Functions (2×2: Neighborhood / Isotropic / Manifold / Manifold-no-whiten)

In [ ]:
LABEL_COLORS = {"Smile": "#2ca02c", "No Smile": "#d62728"}
FLIP_COLOR = "#1a1a1a"
N_ANNOTATE = 0  # images don't need text annotations on scatter


def _build_celeba_legend(ax, mode="neighborhood"):
    handles = [
        Line2D([0], [0], marker='*', color='w', markerfacecolor='gold',
               markeredgecolor='k', markersize=14, label='Anchor'),
        mpatches.Patch(color=LABEL_COLORS['Smile'], label='Smile'),
        mpatches.Patch(color=LABEL_COLORS['No Smile'], label='No Smile'),
    ]
    if mode in ("manifold", "gaussian"):
        handles.append(Line2D([0], [0], marker='o', color='w', markerfacecolor=FLIP_COLOR,
                              markeredgecolor='k', markersize=6, label='Label flip'))
    if mode == "manifold":
        handles.insert(1, Line2D([0], [0], marker='D', color='w', markerfacecolor='#ccc',
                                 markeredgecolor='gray', markersize=5, label='kNN neighbor'))
    ax.legend(handles=handles, loc='upper right', fontsize=7, framealpha=0.9)


def _scatter_samples(ax, anchor_vec, anchor_label, pca_viz, sample_vecs,
                     sample_labels, neighbor_vecs=None, neighbor_labels=None,
                     title="", mode="gaussian"):
    """Shared scatter plot helper for all 4 panels."""
    parts = [anchor_vec.reshape(1, -1)]
    if neighbor_vecs is not None:
        parts.append(neighbor_vecs)
    parts.append(sample_vecs)
    all_pts = np.vstack(parts)
    proj = project_to_local_pca_2d(all_pts, pca_viz)

    p_anchor = proj[0]
    offset = 1

    # Neighbors (diamonds, faint)
    if neighbor_vecs is not None:
        p_nb = proj[offset:offset + len(neighbor_vecs)]
        offset += len(neighbor_vecs)
        for i in range(len(neighbor_vecs)):
            c = LABEL_COLORS.get(neighbor_labels[i], '#333')
            ax.scatter(p_nb[i, 0], p_nb[i, 1], c=c, s=15, alpha=0.2,
                       edgecolors='gray', linewidths=0.2, marker='D')

    # Samples
    p_s = proj[offset:]
    for i in range(len(sample_vecs)):
        same = sample_labels[i] == anchor_label
        c = LABEL_COLORS.get(sample_labels[i], '#333') if same else FLIP_COLOR
        ax.scatter(p_s[i, 0], p_s[i, 1], c=c, s=55, alpha=0.65, marker='o',
                   edgecolors='k', linewidths=0.3)

    # Anchor star
    c = LABEL_COLORS.get(anchor_label, '#333')
    ax.scatter(p_anchor[0], p_anchor[1], c=c, s=400, marker='*',
               edgecolors='k', linewidths=1.5, zorder=10)

    same_count = sum(1 for l in sample_labels if l == anchor_label)
    total = len(sample_labels)
    dist = Counter(sample_labels)
    dist_str = "  ".join(f"{l}:{n}" for l, n in dist.most_common())
    ax.text(0.02, 0.02, dist_str, transform=ax.transAxes, fontsize=7,
            verticalalignment='bottom', bbox=dict(boxstyle='round', fc='wheat', alpha=0.85))

    _build_celeba_legend(ax, mode=mode)
    ax.set_title(f"{title}\nPreserved: {same_count}/{total}", fontsize=11, fontweight='bold')
    ax.set_xlabel('PC1'); ax.set_ylabel('PC2')
    ax.grid(True, alpha=0.2)


print("Scatter helpers defined.")

In [ ]:
def plot_celeba_2x2(anchor_vec, anchor_label, vectors, index, classify_fn,
                    sigma=0.5, knn_k=200, n_samples=100, space_name="pixel"):
    """Generate 2×2 visualization for one sample in a given space.
    
    Args:
        anchor_vec: (D,) embedding vector
        anchor_label: 'Smile' or 'No Smile'
        vectors: all index vectors (N, D) for neighbor lookup
        index: NeighborIndex
        classify_fn: function(vecs) → list of (label, prob)
        space_name: 'pixel' or 'latent' (for titles)
    """
    fig, axes = plt.subplots(2, 2, figsize=(18, 14))

    # ── A) kNN Neighborhood ──
    ax = axes[0, 0]
    nids = query_index(index, k=knn_k + 1, vector=anchor_vec)[1:]
    nb_vecs = vectors[nids]
    nb_labels = [r[0] for r in classify_fn(nb_vecs)]

    pca_viz = fit_local_pca(nb_vecs)
    _scatter_samples(ax, anchor_vec, anchor_label, pca_viz,
                     nb_vecs, nb_labels, title=f"A) kNN Neighborhood ({space_name})",
                     mode="neighborhood")

    # ── B) Isotropic (Gaussian) Smoothing ──
    ax = axes[0, 1]
    noise = np.random.randn(n_samples, anchor_vec.shape[0]).astype(np.float32) * sigma
    iso_vecs = anchor_vec.reshape(1, -1) + noise
    iso_labels = [r[0] for r in classify_fn(iso_vecs)]
    _scatter_samples(ax, anchor_vec, anchor_label, pca_viz,
                     iso_vecs, iso_labels,
                     title=f"B) Isotropic Smoothing σ={sigma} ({space_name})",
                     mode="gaussian")

    # ── C) Manifold Smoothing (with whitening) ──
    ax = axes[1, 0]
    smoother = ManifoldSmoother(sigma=sigma, index=index, knn_k=knn_k, eps_eig=1e-6)
    mani_vecs = smoother.sample_n(anchor_vec, n_samples)
    mani_labels = [r[0] for r in classify_fn(mani_vecs)]
    _scatter_samples(ax, anchor_vec, anchor_label, pca_viz,
                     mani_vecs, mani_labels,
                     neighbor_vecs=nb_vecs, neighbor_labels=nb_labels,
                     title=f"C) Manifold Smoothing σ={sigma} ({space_name})",
                     mode="manifold")

    # ── D) Manifold Smoothing — NO whitening ──
    ax = axes[1, 1]
    local_pca = fit_local_pca(nb_vecs)
    d = min(knn_k, nb_vecs.shape[1])
    anchor_centered = anchor_vec - local_pca.mean
    coeffs = anchor_centered @ local_pca.evecs[:, :d]
    noise_pca = np.random.randn(n_samples, d).astype(np.float32) * sigma
    sample_coeffs = coeffs[None, :] + noise_pca
    nowhiten_vecs = (sample_coeffs @ local_pca.evecs[:, :d].T + local_pca.mean).astype(np.float32)
    nowhiten_labels = [r[0] for r in classify_fn(nowhiten_vecs)]
    _scatter_samples(ax, anchor_vec, anchor_label, pca_viz,
                     nowhiten_vecs, nowhiten_labels,
                     neighbor_vecs=nb_vecs, neighbor_labels=nb_labels,
                     title=f"D) Manifold (no whiten) σ={sigma} ({space_name})",
                     mode="manifold")

    # ── Shared axis limits ──
    all_ax = list(axes.flat)
    xlims = [a.get_xlim() for a in all_ax]
    ylims = [a.get_ylim() for a in all_ax]
    for a in all_ax:
        a.set_xlim(min(lo for lo, _ in xlims), max(hi for _, hi in xlims))
        a.set_ylim(min(lo for lo, _ in ylims), max(hi for _, hi in ylims))

    fig.suptitle(f"{space_name.upper()} SPACE — {anchor_label}  (σ={sigma}, k={knn_k}, n={n_samples})",
                 fontsize=15, fontweight='bold', y=1.01)
    plt.tight_layout()
    return fig


print("plot_celeba_2x2 defined.")

## 6. Decoded Image Grid (show what the noisy samples look like)

In [ ]:
def show_decoded_grid(anchor_vec, anchor_label, vectors, index, classify_fn,
                      to_pil_fn, sigma=0.5, knn_k=200, n_show=5, space_name="pixel"):
    """Show original + kNN neighbors + isotropic samples + manifold samples as images.
    
    4 rows:
      Row 1: Anchor + kNN neighbors
      Row 2: Isotropic noise samples
      Row 3: Manifold (whitened) samples
      Row 4: Manifold (no whiten) samples
    """
    n_cols = n_show + 1
    fig, axes = plt.subplots(4, n_cols, figsize=(3 * n_cols, 12))
    for ax_row in axes:
        for ax in ax_row:
            ax.axis('off')

    row_labels = ['Anchor + kNN', 'Isotropic', 'Manifold (whiten)', 'Manifold (no whiten)']
    for i, label in enumerate(row_labels):
        axes[i, 0].set_title(label, fontsize=10, fontweight='bold', loc='left')

    # Row 0: anchor + neighbors
    axes[0, 0].imshow(to_pil_fn(anchor_vec))
    axes[0, 0].set_title(f"Anchor\n{anchor_label}", fontsize=9)
    nids = query_index(index, k=n_show + 1, vector=anchor_vec)[1:]
    for j, nid in enumerate(nids[:n_show]):
        nb_vec = vectors[int(nid)]
        lbl, prob = classify_fn(nb_vec.reshape(1, -1))[0]
        axes[0, j + 1].imshow(to_pil_fn(nb_vec))
        axes[0, j + 1].set_title(f"NN{j+1}: {lbl}\n{prob:.2f}", fontsize=8)

    # Row 1: isotropic
    iso_noise = np.random.randn(n_show, anchor_vec.shape[0]).astype(np.float32) * sigma
    iso_vecs = anchor_vec.reshape(1, -1) + iso_noise
    for j in range(n_show):
        lbl, prob = classify_fn(iso_vecs[j:j+1])[0]
        color = 'green' if lbl == anchor_label else 'red'
        axes[1, j + 1].imshow(to_pil_fn(iso_vecs[j]))
        axes[1, j + 1].set_title(f"{lbl} {prob:.2f}", fontsize=8, color=color)

    # Row 2: manifold (whitened)
    smoother = ManifoldSmoother(sigma=sigma, index=index, knn_k=knn_k, eps_eig=1e-6)
    mani_vecs = smoother.sample_n(anchor_vec, n_show)
    for j in range(n_show):
        lbl, prob = classify_fn(mani_vecs[j:j+1])[0]
        color = 'green' if lbl == anchor_label else 'red'
        axes[2, j + 1].imshow(to_pil_fn(mani_vecs[j]))
        axes[2, j + 1].set_title(f"{lbl} {prob:.2f}", fontsize=8, color=color)

    # Row 3: manifold no-whiten
    nb_vecs_full = vectors[query_index(index, k=knn_k + 1, vector=anchor_vec)[1:]]
    local_pca = fit_local_pca(nb_vecs_full)
    d = min(knn_k, nb_vecs_full.shape[1])
    coeffs = (anchor_vec - local_pca.mean) @ local_pca.evecs[:, :d]
    noise_pca = np.random.randn(n_show, d).astype(np.float32) * sigma
    nowhiten_vecs = (coeffs[None, :] + noise_pca) @ local_pca.evecs[:, :d].T + local_pca.mean
    nowhiten_vecs = nowhiten_vecs.astype(np.float32)
    for j in range(n_show):
        lbl, prob = classify_fn(nowhiten_vecs[j:j+1])[0]
        color = 'green' if lbl == anchor_label else 'red'
        axes[3, j + 1].imshow(to_pil_fn(nowhiten_vecs[j]))
        axes[3, j + 1].set_title(f"{lbl} {prob:.2f}", fontsize=8, color=color)

    fig.suptitle(f"{space_name.upper()} — Decoded Samples ({anchor_label}, σ={sigma})",
                 fontsize=14, fontweight='bold')
    plt.tight_layout()
    return fig


print("show_decoded_grid defined.")

## 7. Run Visualization — PIXEL SPACE

In [ ]:
SIGMA = 0.5
KNN_K = 200
N_SAMPLES = 100

for sample_idx in selected_idxs:
    sample_idx = int(sample_idx)
    pix_vec = pixel_vectors[sample_idx]
    lbl, prob = classify_pixel_vectors(pix_vec)[0]
    print(f"\n{'='*60}")
    print(f"Sample {sample_idx}: {lbl} (prob={prob:.3f})")
    print(f"{'='*60}")

    # 2×2 PCA scatter
    fig1 = plot_celeba_2x2(
        pix_vec, lbl, pixel_vectors, pixel_index, classify_pixel_vectors,
        sigma=SIGMA, knn_k=KNN_K, n_samples=N_SAMPLES, space_name="pixel")
    plt.show()

    # Decoded image grid
    fig2 = show_decoded_grid(
        pix_vec, lbl, pixel_vectors, pixel_index, classify_pixel_vectors,
        flat_pixel_to_pil, sigma=SIGMA, knn_k=KNN_K, n_show=5, space_name="pixel")
    plt.show()

## 8. Run Visualization — LATENT SPACE

In [ ]:
for sample_idx in selected_idxs:
    sample_idx = int(sample_idx)
    lat_vec = latent_vectors[sample_idx]
    lbl, prob = classify_latent_vectors(lat_vec)[0]
    print(f"\n{'='*60}")
    print(f"Sample {sample_idx}: {lbl} (prob={prob:.3f})  [latent space]")
    print(f"{'='*60}")

    # 2×2 PCA scatter
    fig1 = plot_celeba_2x2(
        lat_vec, lbl, latent_vectors, latent_index, classify_latent_vectors,
        sigma=SIGMA, knn_k=KNN_K, n_samples=N_SAMPLES, space_name="latent")
    plt.show()

    # Decoded image grid (latent → VAE decode → image)
    fig2 = show_decoded_grid(
        lat_vec, lbl, latent_vectors, latent_index, classify_latent_vectors,
        latent_to_pil, sigma=SIGMA, knn_k=KNN_K, n_show=5, space_name="latent")
    plt.show()

## 9. Label Preservation Rate Comparison

In [ ]:
# Quantify label preservation across multiple samples
N_EVAL = 20      # number of anchor samples to test
N_NOISE = 100    # noise samples per anchor

rng_eval = np.random.default_rng(123)
eval_idxs = rng_eval.choice(pixel_vectors.shape[0], size=N_EVAL, replace=False)

rows = []
for space_name, vecs, idx_obj, cls_fn in [
    ("pixel",  pixel_vectors,  pixel_index,  classify_pixel_vectors),
    ("latent", latent_vectors, latent_index, classify_latent_vectors),
]:
    smoother = ManifoldSmoother(sigma=SIGMA, index=idx_obj, knn_k=KNN_K, eps_eig=1e-6)
    for i in eval_idxs:
        vec = vecs[int(i)]
        anchor_lbl = cls_fn(vec)[0][0]

        # Isotropic
        iso = vec.reshape(1, -1) + np.random.randn(N_NOISE, vec.shape[0]).astype(np.float32) * SIGMA
        iso_lbls = [r[0] for r in cls_fn(iso)]
        iso_pres = sum(1 for l in iso_lbls if l == anchor_lbl) / N_NOISE

        # Manifold
        mani = smoother.sample_n(vec, N_NOISE)
        mani_lbls = [r[0] for r in cls_fn(mani)]
        mani_pres = sum(1 for l in mani_lbls if l == anchor_lbl) / N_NOISE

        rows.append({'space': space_name, 'label': anchor_lbl,
                     'isotropic': iso_pres, 'manifold': mani_pres})

import pandas as pd
df_pres = pd.DataFrame(rows)
summary = df_pres.groupby(['space', 'label'])[['isotropic', 'manifold']].mean()
summary['Δ'] = summary['manifold'] - summary['isotropic']
print("LABEL PRESERVATION RATE (higher = better)")
print("=" * 55)
display(summary.round(3))

# Bar chart
fig, ax = plt.subplots(figsize=(10, 5))
summary_reset = summary.reset_index()
x = np.arange(len(summary_reset))
w = 0.35
ax.bar(x - w/2, summary_reset['isotropic'], w, label='Isotropic', color='#e41a1c', alpha=0.7)
ax.bar(x + w/2, summary_reset['manifold'], w, label='Manifold', color='#4daf4a', alpha=0.7)
ax.set_xticks(x)
ax.set_xticklabels([f"{r.space}\n{r.label}" for _, r in summary_reset.iterrows()])
ax.set_ylabel('Label Preservation Rate')
ax.set_title(f'CelebA: Isotropic vs Manifold Smoothing (σ={SIGMA}, k={KNN_K})')
ax.legend()
ax.set_ylim(0, 1.05)
ax.grid(True, alpha=0.2, axis='y')
plt.tight_layout()
plt.show()